# OLEFINS (CONCAWE proxy) – Oléfines totales (raffinerie / FCC-proxy)

Ce notebook lit un YAML de configuration, charge les fichiers CONCAWE refinery, reconstruit un proxy d'oléfines basé sur la route FCC, puis génère 3 scénarios (constant / +3% / -1%) avec substitution stricte.


In [9]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import yaml


In [10]:
# =============================================================================
# LOAD YAML CONFIG
# =============================================================================

CONFIG_PATH = Path("olefins_concawe_config.yaml")  # <-- adapter si besoin

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

YEAR_START = int(CFG["years"]["start"])
YEAR_END = int(CFG["years"]["end"])
REFERENCE_YEAR = int(CFG["years"]["reference_year"])
YEARS = np.arange(YEAR_START, YEAR_END + 1)

UNITS_CFG = CFG.get("units", {})
SCALE_TO_TONNES = float(UNITS_CFG.get("scale_to_tonnes", 1.0))

REF_CFG = CFG["refinery_inputs"]
OLE_CFG = CFG["olefins_proxy"]
DEMAND_CFG = CFG["demand_scenarios"]
OUT_CFG = CFG["output"]
CHECKS = CFG["checks"]

SPLIT_CFG = CFG.get("nonrefinery_split", {})
H2_CFG = CFG.get("h2_intensities", {})

# Split parts
E_SHARE = float(SPLIT_CFG.get("e_olefins_share", 0.70))
BIOE_SHARE = float(SPLIT_CFG.get("bio_e_olefins_share", 0.10))
BIO_SHARE = float(SPLIT_CFG.get("bio_olefins_share", 0.20))
if abs((E_SHARE + BIOE_SHARE + BIO_SHARE) - 1.0) > 1e-9:
    raise ValueError("nonrefinery_split shares must sum to 1.0")

# H2 intensities (tH2 / t olefins)
H2_PER_T_E = float(H2_CFG.get("h2_per_t_e_olefins", 0.4286))
H2_PER_T_REFINERY = float(H2_CFG.get("h2_per_t_refinery_proxy_olefins", 0.05))
H2_PER_T_BIO = float(H2_CFG.get("h2_per_t_bio_olefins", 0.0))

OUT_DIR = Path(OUT_CFG["base_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_COLS = OUT_CFG.get("columns", {})



In [11]:
# =============================================================================
# FUNCTIONS
# =============================================================================


def read_refinery(path: Path) -> pd.DataFrame:
    """Read refinery file using YAML schema."""
    df = pd.read_csv(path)

    schema = REF_CFG["schema"]
    df = df.rename(columns={
        schema["country_column"]: "country",
        schema["year_column"]: "year",
        schema["unit_column"]: "unit",
        schema["feed_column"]: "unit_feed",
    })

    required = {"country", "year", "unit", "unit_feed"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}")

    df = df.copy()
    df["country"] = df["country"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype(int)
    df["unit_feed"] = pd.to_numeric(df["unit_feed"], errors="coerce").fillna(0.0) * SCALE_TO_TONNES

    return df[df["year"].isin(YEARS)]


def compute_olefins_proxy(df_ref: pd.DataFrame) -> pd.DataFrame:
    """Olefins_proxy_refinery(t) = coeff * feed(proxy_unit)."""
    proxy_unit = OLE_CFG["fcc_proxy_unit"]
    coeff = float(OLE_CFG["coeff_fcc_to_olefins"])

    g = (df_ref.groupby(["country", "year", "unit"], as_index=False)["unit_feed"].sum())
    fcc = g[g["unit"] == proxy_unit].copy()

    if fcc.empty:
        raise ValueError(f"FCC proxy unit not found: {proxy_unit}")

    out = (
        fcc.groupby(["country", "year"], as_index=False)["unit_feed"]
        .sum()
        .rename(columns={"unit_feed": "olefins_proxy_refinery_t_per_yr"})
    )
    out["olefins_proxy_refinery_t_per_yr"] = coeff * out["olefins_proxy_refinery_t_per_yr"]
    return out


def build_olefins_scenario(olefins_proxy: pd.DataFrame, growth_rate: float) -> pd.DataFrame:
    """Strict 'substitution' toward a total olefins trajectory + split + H2."""
    rows = []

    for country, g in olefins_proxy.groupby("country"):
        g = g.set_index("year").sort_index()

        if REFERENCE_YEAR not in g.index:
            continue

        ole_ref = float(g.loc[REFERENCE_YEAR, "olefins_proxy_refinery_t_per_yr"])

        for year in YEARS:
            if year not in g.index:
                continue

            ole_proxy = float(g.loc[year, "olefins_proxy_refinery_t_per_yr"])
            ole_total = ole_ref * ((1.0 + growth_rate) ** (year - REFERENCE_YEAR))

            ole_refinery = min(ole_proxy, ole_total)
            ole_nonref = ole_total - ole_refinery

            # Split non-refinery
            ole_e = E_SHARE * ole_nonref
            ole_bioe = BIOE_SHARE * ole_nonref
            ole_bio = BIO_SHARE * ole_nonref

            # H2 demands
            h2_refinery = ole_refinery * H2_PER_T_REFINERY
            h2_nonref = (ole_e + ole_bioe) * H2_PER_T_E + ole_bio * H2_PER_T_BIO
            h2_total = h2_refinery + h2_nonref

            rows.append({
                "country": country,
                "year": int(year),

                "olefins_proxy_refinery_t_per_yr": ole_proxy,
                "olefins_refinery_t_per_yr": ole_refinery,
                "olefins_total_t_per_yr": ole_total,
                "olefins_nonrefinery_t_per_yr": ole_nonref,

                "olefins_e_t_per_yr": ole_e,
                "olefins_bio_e_t_per_yr": ole_bioe,
                "olefins_bio_t_per_yr": ole_bio,

                "h2_refinery_t_per_yr": h2_refinery,
                "h2_nonrefinery_t_per_yr": h2_nonref,
                "h2_total_t_per_yr": h2_total,
            })

    return pd.DataFrame(rows)


def run_checks(df: pd.DataFrame) -> None:
    if df.empty:
        raise ValueError("Empty output dataframe (no countries processed).")

    if CHECKS.get("non_negative_flows", True):
        cols = [
            "olefins_proxy_refinery_t_per_yr",
            "olefins_refinery_t_per_yr",
            "olefins_total_t_per_yr",
            "olefins_nonrefinery_t_per_yr",
            "olefins_e_t_per_yr",
            "olefins_bio_e_t_per_yr",
            "olefins_bio_t_per_yr",
            "h2_refinery_t_per_yr",
            "h2_nonrefinery_t_per_yr",
            "h2_total_t_per_yr",
        ]
        if (df[cols] < -1e-9).any().any():
            raise ValueError("Negative flows detected.")

    if CHECKS.get("enforce_mass_balance", True):
        # olefins_total = olefins_refinery + olefins_nonrefinery
        diff1 = df["olefins_total_t_per_yr"] - (df["olefins_refinery_t_per_yr"] + df["olefins_nonrefinery_t_per_yr"])
        if (diff1.abs() > 1e-6).any():
            raise ValueError("Mass balance violation detected (total vs refinery+nonrefinery).")

        # olefins_nonrefinery = e + bio-e + bio
        diff2 = df["olefins_nonrefinery_t_per_yr"] - (
            df["olefins_e_t_per_yr"] + df["olefins_bio_e_t_per_yr"] + df["olefins_bio_t_per_yr"]
        )
        if (diff2.abs() > 1e-6).any():
            raise ValueError("Mass balance violation detected (nonrefinery split).")

    if CHECKS.get("full_year_coverage", True):
        expected = len(YEARS)
        for country, g in df.groupby("country"):
            if len(g["year"].unique()) != expected:
                raise ValueError(f"Incomplete year coverage for {country}")


In [12]:
# =============================================================================
# MAIN
# =============================================================================

for ref_name, ref_info in REF_CFG["scenarios"].items():
    print(f"\nProcessing refinery scenario: {ref_name}")

    df_ref = read_refinery(Path(ref_info["path"]))
    ole_proxy = compute_olefins_proxy(df_ref)

    for demand_name, demand_info in DEMAND_CFG.items():
        growth = float(demand_info["annual_growth_rate"])
        print(f"  → Building olefins scenario: {demand_name}_{ref_name}")

        df_out = build_olefins_scenario(ole_proxy, growth)
        run_checks(df_out)

        # Add scenario tags
        df_out["refinery_scenario"] = ref_name
        df_out["demand_scenario"] = demand_name

        # Apply output column naming from YAML
        rename_map = {
            "refinery_scenario": OUT_COLS.get("refinery_scenario", "refinery_scenario"),
            "demand_scenario": OUT_COLS.get("demand_scenario", "demand_scenario"),

            "olefins_proxy_refinery_t_per_yr": OUT_COLS.get("olefins_proxy_refinery", "olefins_proxy_refinery_t_per_yr"),
            "olefins_refinery_t_per_yr": OUT_COLS.get("olefins_refinery", "olefins_refinery_t_per_yr"),
            "olefins_total_t_per_yr": OUT_COLS.get("olefins_total", "olefins_total_t_per_yr"),
            "olefins_nonrefinery_t_per_yr": OUT_COLS.get("olefins_nonrefinery", "olefins_nonrefinery_t_per_yr"),

            "olefins_e_t_per_yr": OUT_COLS.get("olefins_e", "olefins_e_t_per_yr"),
            "olefins_bio_e_t_per_yr": OUT_COLS.get("olefins_bio_e", "olefins_bio_e_t_per_yr"),
            "olefins_bio_t_per_yr": OUT_COLS.get("olefins_bio", "olefins_bio_t_per_yr"),

            "h2_refinery_t_per_yr": OUT_COLS.get("h2_refinery", "h2_refinery_t_per_yr"),
            "h2_nonrefinery_t_per_yr": OUT_COLS.get("h2_nonrefinery", "h2_nonrefinery_t_per_yr"),
            "h2_total_t_per_yr": OUT_COLS.get("h2_total", "h2_total_t_per_yr"),
        }
        df_out = df_out.rename(columns=rename_map)

        # Output directory structure: base_dir/refinery_scenario/demand_scenario/olefins.csv
        out_path = OUT_DIR / ref_name / demand_name
        out_path.mkdir(parents=True, exist_ok=True)

        df_out.to_csv(out_path / OUT_CFG["output_file_name"], index=False)

print("\nAll olefins CONCAWE scenarios successfully written.")


Processing refinery scenario: more-molecule
  → Building olefins scenario: constant-demand_more-molecule
  → Building olefins scenario: growth-3pct_more-molecule
  → Building olefins scenario: decline-1pct_more-molecule

Processing refinery scenario: max-electron
  → Building olefins scenario: constant-demand_max-electron
  → Building olefins scenario: growth-3pct_max-electron
  → Building olefins scenario: decline-1pct_max-electron

All olefins CONCAWE scenarios successfully written.
